# Língua — vetorização da etapa 09 no Google Colab

1. Baixe o notebook e o ZIP da mesma versão do Língua. Abra este notebook no Colab.
2. Execute as células em ordem e envie o ZIP exportado pelo aplicativo.
3. CPU funciona; para GPU, use **Ambiente de execução → Alterar tipo de ambiente de execução**.

O ZIP contém o texto e a origem linguística completos: o upload envia esses dados ao Google Colab. Execute somente o notebook recebido do seu Língua; hashes verificam integridade, não a confiança no autor. O modelo será baixado do Hugging Face e precisa de internet, memória e espaço disponíveis. Esta execução não refaz as etapas 01–08.


## 1. Enviar e validar o pacote

Nenhum módulo do ZIP é importado antes de conferir estrutura, tamanhos e hashes do código confiável.


In [ ]:
from hashlib import sha256
from io import BytesIO
import json, os, re, shutil, stat, sys, tempfile
from pathlib import Path
from zipfile import BadZipFile, ZIP_DEFLATED, ZIP_STORED, ZipFile
from google.colab import files

class ErroPacoteColab(ValueError):
    pass

FORMATO = 'lingua_etapa09_colab'
VERSAO = '1.0.0'
MAX_BYTES_ZIP = 67108864
MAX_BYTES_ARQUIVO = 100663296
MAX_BYTES_PACOTE = 134217728
ARQUIVOS_PACOTE = frozenset(['LEIA-ME.txt', 'anotacao.py', 'catalogo_regras.py', 'configuracao.json', 'contexto.json', 'contratos_vetorizacao.py', 'embeddings_e5.py', 'exportacao_colab.py', 'lingua_etapa09_colab.ipynb', 'manifesto.json', 'persistencia_vetores.py', 'preparacao.py', 'regras_detectores.py', 'regras_linguisticas.py', 'requirements-embeddings.txt', 'segmentacao.py', 'sintaxe_entidades.py', 'unidades_contexto.py', 'vetorizacao.py'])
HASHES_CODIGO_CONFIAVEIS = {'preparacao.py': 'c87a396562b6af333e9262753cdde026b7f750b5d74a1e3d05b1200d422d5d1f', 'segmentacao.py': 'ad44e51b2eda51b261fb85d2043adefe8f05c1430b55522a1bb65dc6318da64b', 'anotacao.py': 'c2375c95ab3b09a868445d8e5be3c0fbe0fe7b96063e6c49b0b93d9dbb86154a', 'sintaxe_entidades.py': '2a07181287c9d952e957af754e5777c1e6e9f484eba12699af6460686a1b254c', 'catalogo_regras.py': 'e7c1e2224963fdc3de4ab9b2dd4db68f8856fb32668868a83ebc39bd96624f9a', 'regras_detectores.py': 'adbaa55bfd1337d080a25169fb947a44a1af3221dd645a3c91854579bc2bf8e5', 'regras_linguisticas.py': '1b9063c8e60b45e3bf68238cf9f46ef515a4cd49d98a3f1e64074f7fb96a678b', 'unidades_contexto.py': '5ebc064038acbc4aa22d22651360b02e9d464e4c58bf4fb14d3c9337f3007e53', 'contratos_vetorizacao.py': 'f0474f8ce85ee8458ea0353dc56fb580e6afc1f7274b769f09b5b2edb6ff653e', 'vetorizacao.py': '0e48788afd82135e1cdf9f790f1ea0ccecd09592b14ece5f89a66d03bff7fcec', 'embeddings_e5.py': 'a93853677ae1ff5223a3af3571a605fa40ee46e4e7e01185e8f7b8de908496c5', 'persistencia_vetores.py': '9e83f56e8a540f7261a93ec9806176f32a8705c36f2f1e1b79061daab069edfd', 'exportacao_colab.py': '722344e7c0be9821115bc6511f011a54d9b3d926a5a9107b836dfe3135faa11d', 'requirements-embeddings.txt': '8432e9e6553e595d15e32a4181958fd8a124b21ea0f3d760bd6371f7006ac741'}

def _json_bytes(dados):
    def objeto(pares):
        valor = {}
        for chave, conteudo in pares:
            if chave in valor:
                raise ErroPacoteColab("JSON contém chave duplicada.")
            valor[chave] = conteudo
        return valor

    def constante(_valor):
        raise ErroPacoteColab("JSON contém número não finito.")

    try:
        return json.loads(dados.decode("utf-8"), object_pairs_hook=objeto,
                          parse_constant=constante)
    except (UnicodeError, ValueError, TypeError, RecursionError) as erro:
        if isinstance(erro, ErroPacoteColab):
            raise
        raise ErroPacoteColab("Arquivo JSON inválido ou fora dos limites.") from erro

def _validar_zip_bruto(dados):
    """Somente stdlib; esta função também é incorporada ao notebook confiável."""
    if type(dados) is not bytes or len(dados) > MAX_BYTES_ZIP:
        raise ErroPacoteColab("ZIP deve conter bytes e respeitar o limite de 64 MiB.")
    try:
        with ZipFile(BytesIO(dados)) as arquivo:
            membros = arquivo.infolist()
            nomes = [m.filename for m in membros]
            if len(nomes) != len(set(nomes)):
                raise ErroPacoteColab("ZIP contém membros duplicados.")
            if set(nomes) != ARQUIVOS_PACOTE:
                raise ErroPacoteColab("ZIP contém arquivos ausentes, extras ou caminhos inválidos.")
            tamanho = 0
            for membro in membros:
                nome = membro.filename
                modo = stat.S_IFMT(membro.external_attr >> 16)
                if ("/" in nome or "\\" in nome or nome in (".", "..")
                        or membro.is_dir() or modo not in (0, stat.S_IFREG)
                        or membro.flag_bits & 1
                        or membro.compress_type not in (ZIP_STORED, ZIP_DEFLATED)):
                    raise ErroPacoteColab("ZIP contém caminho, tipo ou compressão não permitidos.")
                if membro.file_size > MAX_BYTES_ARQUIVO:
                    raise ErroPacoteColab("Arquivo do ZIP excede 96 MiB.")
                tamanho += membro.file_size
                if tamanho > MAX_BYTES_PACOTE:
                    raise ErroPacoteColab("ZIP descompactado excede 128 MiB.")
            payloads = {m.filename: arquivo.read(m) for m in membros}
            if any(len(payloads[m.filename]) != m.file_size for m in membros):
                raise ErroPacoteColab("Tamanho real diverge do diretório do ZIP.")
    except (BadZipFile, OSError, EOFError, RuntimeError, NotImplementedError) as erro:
        raise ErroPacoteColab("ZIP inválido ou não legível.") from erro
    manifesto = _json_bytes(payloads["manifesto.json"])
    if (type(manifesto) is not dict
            or set(manifesto) != {"formato", "versao", "origem", "arquivos"}
            or manifesto["formato"] != FORMATO or manifesto["versao"] != VERSAO):
        raise ErroPacoteColab("Contrato do manifesto não reconhecido.")
    origem = manifesto["origem"]
    if (type(origem) is not dict
            or set(origem) != {"documento_id", "contexto_execucao_id", "contexto_sha256"}
            or not ((type(origem["documento_id"]) is str and origem["documento_id"].strip())
                    or (type(origem["documento_id"]) is int and origem["documento_id"] >= 0))
            or type(origem["contexto_execucao_id"]) is not str or not origem["contexto_execucao_id"].strip()
            or type(origem["contexto_sha256"]) is not str
            or re.fullmatch(r"[0-9a-f]{64}", origem["contexto_sha256"]) is None):
        raise ErroPacoteColab("Identificação da origem ausente ou inválida.")
    hashes = manifesto["arquivos"]
    if type(hashes) is not dict or set(hashes) != ARQUIVOS_PACOTE - {"manifesto.json"}:
        raise ErroPacoteColab("Manifesto não cobre todos os arquivos do pacote.")
    for nome, metadados in hashes.items():
        conteudo = payloads[nome]
        if (type(metadados) is not dict or set(metadados) != {"bytes", "sha256"}
                or type(metadados["bytes"]) is not int or metadados["bytes"] != len(conteudo)
                or type(metadados["sha256"]) is not str
                or metadados["sha256"] != sha256(conteudo).hexdigest()):
            raise ErroPacoteColab(f"Tamanho ou hash divergente: {nome}.")
    return payloads, manifesto

def _extrair_payloads(payloads, destino):
    destino = Path(destino)
    if destino.exists() or destino.is_symlink() or not destino.parent.is_dir():
        raise ErroPacoteColab("Destino deve ser novo, em uma pasta existente.")
    temporario = Path(tempfile.mkdtemp(prefix=".lingua-colab-", dir=destino.parent))
    try:
        for nome, conteudo in payloads.items():
            # Os nomes são planos e pertencem à lista fechada já validada.
            (temporario / nome).write_bytes(conteudo)
        if destino.exists() or destino.is_symlink():
            raise ErroPacoteColab("Destino deixou de estar disponível.")
        os.rename(temporario, destino)
    finally:
        if temporario.exists():
            shutil.rmtree(temporario)

enviados = files.upload()
if len(enviados) != 1:
    raise ValueError('Envie somente o ZIP exportado pelo Língua.')
nome_zip, dados_zip = next(iter(enviados.items()))
payloads, manifesto = _validar_zip_bruto(dados_zip)
for nome, hash_esperado in HASHES_CODIGO_CONFIAVEIS.items():
    if sha256(payloads[nome]).hexdigest() != hash_esperado:
        raise ValueError('Código diferente do notebook confiável: ' + nome + '. Baixe novamente notebook e ZIP da mesma versão do Língua.')
PASTA_PACOTE = Path(tempfile.mkdtemp(prefix='lingua-colab-')) / 'pacote'
_extrair_payloads(payloads, PASTA_PACOTE)
sys.path.insert(0, str(PASTA_PACOTE))
for nome_modulo in ['preparacao', 'segmentacao', 'anotacao', 'sintaxe_entidades', 'catalogo_regras', 'regras_detectores', 'regras_linguisticas', 'unidades_contexto', 'contratos_vetorizacao', 'vetorizacao', 'embeddings_e5', 'persistencia_vetores', 'exportacao_colab']:
    sys.modules.pop(nome_modulo, None)
from exportacao_colab import validar_pacote_colab, executar_pacote_colab, exportar_resultado_colab
validar_pacote_colab(dados_zip)
del dados_zip, enviados, payloads
print('Origem validada:', manifesto['origem']['contexto_execucao_id'])


## 2. Configurar execução

Escolha o dispositivo e ajuste a finalidade, o lote ou a agregação. GPU é opcional.


In [ ]:
# Escolha CPU ou GPU sem alterar a origem da etapa 08.
DISPOSITIVO = "auto"  # "auto", "cpu" ou "cuda"
PRECISAO = "float32"  # "float16" somente com GPU
INDICE_TORCH_CUDA = "https://download.pytorch.org/whl/cu124"

# A finalidade, o lote e a agregação podem ser ajustados antes da inferência.
CONFIGURACAO = json.loads((PASTA_PACOTE / "configuracao.json").read_text(encoding="utf-8"))
# CONFIGURACAO["perfil"] = "recuperacao"  # "similaridade" ou "consulta"
# CONFIGURACAO["tamanho_lote"] = 4
# CONFIGURACAO["max_tokens"] = 512
# CONFIGURACAO["agregar"] = True
print(CONFIGURACAO)


## 3. Instalar dependências opcionais

A distribuição CPU evita baixar bibliotecas CUDA quando não forem usadas.


In [ ]:
import importlib.metadata
import subprocess

if DISPOSITIVO not in ("auto", "cpu", "cuda"):
    raise ValueError("DISPOSITIVO deve ser auto, cpu ou cuda.")
if PRECISAO not in ("float32", "float16"):
    raise ValueError("PRECISAO deve ser float32 ou float16.")
gpu_disponivel = False
if shutil.which("nvidia-smi"):
    gpu_disponivel = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=10,
    ).returncode == 0
dispositivo_efetivo = ("cuda" if gpu_disponivel else "cpu") if DISPOSITIVO == "auto" else DISPOSITIVO
if dispositivo_efetivo == "cuda" and not gpu_disponivel:
    raise ValueError("GPU solicitada mas indisponível. Selecione CPU ou ative GPU no Colab.")
if dispositivo_efetivo == "cpu" and PRECISAO != "float32":
    raise ValueError("CPU exige PRECISAO=float32.")
indice_torch = INDICE_TORCH_CUDA if dispositivo_efetivo == "cuda" else "https://download.pytorch.org/whl/cpu"
def versao_instalada(nome):
    try:
        return importlib.metadata.version(nome)
    except importlib.metadata.PackageNotFoundError:
        return ""

# Transformers pode importar torchvision mesmo com um modelo apenas de texto.
# Ajuste opcionais já presentes para versões compatíveis com o PyTorch fixado.
versoes_torch = {"torch": "2.6.0"}
for nome, versao in (("torchvision", "0.21.0"), ("torchaudio", "2.6.0")):
    if versao_instalada(nome):
        versoes_torch[nome] = versao
precisa_instalar = any(versao_instalada(nome).split("+")[0] != versao
                      for nome, versao in versoes_torch.items())
precisa_instalar |= dispositivo_efetivo == "cuda" and "+cpu" in versao_instalada("torch")
if precisa_instalar:
    if any(nome in sys.modules for nome in versoes_torch):
        raise RuntimeError("Reinicie a sessão do Colab antes de instalar as versões fixadas de PyTorch e opcionais.")
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *[nome + "==" + versao for nome, versao in versoes_torch.items()],
                    "--index-url", indice_torch], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PASTA_PACOTE / "requirements-embeddings.txt")], check=True)
import torch
from transformers import AutoModel, AutoTokenizer
if dispositivo_efetivo == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("PyTorch não disponibilizou CUDA. Verifique o índice CUDA ou selecione CPU.")
print("PyTorch carregado:", torch.__version__)
print("Dependências preparadas; dispositivo:", dispositivo_efetivo)


## 4. Preparar o E5 em uma revisão oficial fixa

O download pode levar alguns minutos. Nenhuma revisão provisória é usada.


In [ ]:
from embeddings_e5 import preparar_configuracao, criar_gerador_padrao

# "main" é resolvida agora para o SHA oficial; a inferência usa somente esse SHA.
config_modelo = preparar_configuracao(
    modelo_id="intfloat/multilingual-e5-large", revisao="main",
    dispositivo=dispositivo_efetivo, precisao=PRECISAO, baixar=True,
    caminho_config=PASTA_PACOTE / "instance" / "embeddings_modelo.json",
)
gerador = criar_gerador_padrao(caminho_config=PASTA_PACOTE / "instance" / "embeddings_modelo.json")
print("Revisão oficial fixa:", config_modelo["revisao"])


## 5. Vetorizar e validar

Gera vetores do foco, da janela, de cada parágrafo e do documento integral. São preservadas a origem integral, as seleções contextuais, os blocos e a configuração efetiva.


In [ ]:
# Revalida toda a origem 08. Não executa novamente as etapas 01–08.
resultado = executar_pacote_colab(PASTA_PACOTE, gerador=gerador, configuracao=CONFIGURACAO)
from vetorizacao import validar_vetorizacao
validar_vetorizacao(resultado)  # Validação integral, sem executar o modelo.
print(resultado["validacao"])


## 6. Baixar os resultados

O ZIP contém `vetorizacao.json` completo, vetores float32 e manifestos com hashes. A sessão do Colab é temporária: baixe os resultados antes de encerrá-la.


In [ ]:
PASTA_RESULTADOS = PASTA_PACOTE.parent / "resultados"
PASTA_RESULTADOS.mkdir(exist_ok=True)
json_resultado = PASTA_RESULTADOS / "vetorizacao.json"
zip_resultado = PASTA_RESULTADOS / "vetorizacao-colab.zip"
json_resultado.write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
zip_resultado.write_bytes(exportar_resultado_colab(resultado))
files.download(str(zip_resultado))  # Inclui JSON completo, manifesto e vetores binários.
# Para baixar também o JSON separado: files.download(str(json_resultado))
